In [ ]:
"""
=============================================================================
ÉTAPE 2 — Modèle ARX Hurdle Hiérarchique avec Hétéroscédasticité Géographique
=============================================================================

ARCHITECTURE (3 composants) :
  A. Hurdle (Bernoulli) : logit(P) = α_d + β_D·D + β_LB·LB + β_int·D×LB + β_lag·lag
  B. Volume  (ARX)      : log(flow) ~ N(μ_{d,t} + φ_d·(lag - μ_{d,t-1}), σ_d)
                          μ_{d,t} = α_{V,d} + X·β_grav + β_gdp·log_gdpcap_o + β_rich·is_rich_o
  C. Variance (Géo)     : σ_d ~ f(σ_cluster[continent_origine[d]])
=============================================================================
"""

import pandas as pd
import numpy as np
from cmdstanpy import CmdStanModel
import matplotlib.pyplot as plt
import arviz as az
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)


# =============================================================================
# 1. CHARGEMENT ET FILTRAGE
# =============================================================================

DATA_PATH = "../data/data_final/DF_GRAVITY_sans_NaN.csv"
df_main = pd.read_csv(DATA_PATH)

PAYS_STABLES    = ['FRA', 'USA', 'ESP', 'CAN', 'MEX']
PAYS_CHAOTIQUES = ['DZA', 'MMR', 'RWA', 'HTI', 'ZAF', 'NER']
PAYS_TEST       = PAYS_STABLES + PAYS_CHAOTIQUES

df = df_main[
    df_main['orig'].isin(PAYS_TEST) &
    df_main['dest'].isin(PAYS_TEST) &
    (df_main['orig'] != df_main['dest'])
].copy()

df = df.sort_values(['orig', 'dest', 'year']).reset_index(drop=True)
print(f"Lignes après filtrage 11 pays : {len(df)}")


# =============================================================================
# 2. MAPPING CONTINENT (Composant C — Hétéroscédasticité géographique)
#    1=Europe, 2=Am.Nord, 3=Afrique, 4=Am.Sud, 5=Asie, 6=Océanie
# =============================================================================

CONTINENT_MAP = {
    'FRA': 1, 'ESP': 1,                   # Europe
    'USA': 2, 'CAN': 2, 'MEX': 2,         # Amérique du Nord / Centrale
    'DZA': 3, 'NER': 3, 'RWA': 3,         # Afrique
    'ZAF': 3,                              # Afrique du Sud
    'HTI': 4,                              # Amérique du Sud / Caraïbes
    'MMR': 5,                              # Asie
}

df['continent_orig'] = df['orig'].map(CONTINENT_MAP)
assert df['continent_orig'].notna().all(), \
    "ERREUR : certains pays ne sont pas dans CONTINENT_MAP"

K_clusters = 6   # Nombre de continents


# =============================================================================
# 3. CONSTRUCTION DES VARIABLES DÉRIVÉES
# =============================================================================

# --- is_migration ---
if 'is_migration' not in df.columns:
    df['is_migration'] = (df['flow'] > 0).astype(int)

# --- Seuil PIB (Composant B) ---
# Le Random Forest a identifié un saut brutal à log_gdpcap_o_lag ≈ 2.9
SEUIL_LOG_GDP = 2.9
df['is_rich_o'] = (df['log_gdpcap_o_lag'] > SEUIL_LOG_GDP).astype(float)

# --- Variable d'interaction Hurdle (Composant A) ---
# D_ij est la distance brute ; on utilise log(D_ij) pour le hurdle
# LB_ij est déjà binaire
# L'interaction est log(D_ij) * LB_ij
df['log_D_ij']      = np.log(df['D_ij'].replace(0, np.nan))
df['logD_times_LB'] = df['log_D_ij'] * df['LB_ij']

# --- log(flow) pour la partie volume ---
df['log_flow'] = np.where(df['flow'] > 0, np.log(df['flow']), np.nan)


# =============================================================================
# 4. CONSTRUCTION DES LAGS PAR DYADE
# =============================================================================

df = df.sort_values(['orig', 'dest', 'year']).reset_index(drop=True)
df['dyad'] = df['orig'] + "_" + df['dest']

# Lags
df['is_mig_lag']     = df.groupby('dyad')['is_migration'].shift(1)
df['log_flow_lag']   = df.groupby('dyad')['log_flow'].shift(1)

# mu_{d,t-1} a besoin des covariables au temps t-1 pour l'équation ARX complète.
# En pratique, pour l'AR(1), on utilise log_flow_lag comme proxy de μ_{d,t-1}.
# (Approche standard : on ne lag pas toutes les covariables, seulement le flux.)

# Suppression des premières années (lag indisponible)
df = df.dropna(subset=['is_mig_lag']).reset_index(drop=True)
print(f"Après suppression premières années : {len(df)} lignes")


# =============================================================================
# 5. DÉFINITION DES VARIABLES PAR COMPOSANT
# =============================================================================

# --- Variables Hurdle (Composant A) ---
HURDLE_VARS = ['log_D_ij', 'LB_ij', 'logD_times_LB']
# Note : is_mig_lag est traité séparément (coefficient scalaire global)

# --- Variables Gravité (Composant B) ---
GRAVITY_VARS = [
    'P_it',       # Population origine (sera log-transformée)
    'P_jt',       # Population destination
    # D_ij déjà en log dans log_D_ij — on l'ajoute directement en log
    'PSR_i', 'PSR_j',
    'IMR_it', 'IMR_jt',
    'urban_it', 'urban_jt',
    'LA_i', 'LA_j',
    'LL_i', 'LL_j',           # Variables binaires (pas de log)
    'LB_ij', 'OL_ij', 'COL_ij',
    't_2000', 't_2000_sq',
    'log_gdpcap_d_lag',       # Déjà en log dans le dataset
]

# Variables ML (Composant B — seuil PIB)
ML_VARS = ['log_gdpcap_o_lag', 'is_rich_o']

# Variables continues à log-transformer
LOG_TRANSFORM = {
    'P_it': 'log_P_it', 'P_jt': 'log_P_jt',
    'PSR_i': 'log_PSR_i', 'PSR_j': 'log_PSR_j',
    'IMR_it': 'log_IMR_it', 'IMR_jt': 'log_IMR_jt',
    'urban_it': 'log_urban_it', 'urban_jt': 'log_urban_jt',
    'LA_i': 'log_LA_i', 'LA_j': 'log_LA_j',
}

for raw, logged in LOG_TRANSFORM.items():
    df[logged] = np.log(df[raw].replace(0, np.nan))

# Colonnes finales de la matrice X_volume
X_VOL_COLS = (
    ['log_P_it', 'log_P_jt', 'log_D_ij',
     'log_PSR_i', 'log_PSR_j',
     'log_IMR_it', 'log_IMR_jt',
     'log_urban_it', 'log_urban_jt',
     'log_LA_i', 'log_LA_j',
     'LL_i', 'LL_j',
     'LB_ij', 'OL_ij', 'COL_ij',
     't_2000', 't_2000_sq',
     'log_gdpcap_d_lag']
    + ML_VARS
)

K_grav = len(X_VOL_COLS)
K_h    = len(HURDLE_VARS)   # Nombre de covariables Hurdle (hors lag)

print(f"\nMatrice X_volume : {K_grav} colonnes")
print(f"  {X_VOL_COLS}")
print(f"Matrice X_hurdle : {K_h} colonnes + coefficient β_lag")
print(f"  {HURDLE_VARS} + ['is_mig_lag']")


# =============================================================================
# 6. SÉPARATION HURDLE / VOLUME ET DROPNA RIGOUREUX
# =============================================================================

# --- HURDLE : toutes les observations (flow=0 inclus) ---
HURDLE_REQUIRED = HURDLE_VARS + ['is_mig_lag', 'is_migration', 'dyad', 'continent_orig']
df_hurdle = df.dropna(subset=HURDLE_REQUIRED).copy().reset_index(drop=True)

# Vérification des infinis dans la matrice hurdle
X_h_check = df_hurdle[HURDLE_VARS].values
assert not np.any(np.isnan(X_h_check)), "NaN dans X_hurdle après dropna !"
assert not np.any(np.isinf(X_h_check)), "Inf dans X_hurdle !"

N_h = len(df_hurdle)
print(f"\nÉtape 1 (Hurdle) : {N_h} observations")

# --- VOLUME : flow > 0 ET log_flow_lag disponible ---
VOLUME_REQUIRED = X_VOL_COLS + ['log_flow', 'log_flow_lag', 'dyad', 'continent_orig']
df_volume = (
    df[df['flow'] > 0]
    .dropna(subset=VOLUME_REQUIRED)
    .copy()
    .reset_index(drop=True)
)

# Vérification stricte
X_v_check = df_volume[X_VOL_COLS].values
assert not np.any(np.isnan(X_v_check)),  "NaN dans X_volume après dropna !"
assert not np.any(np.isinf(X_v_check)),  "Inf dans X_volume !"
assert not np.any(np.isnan(df_volume['log_flow'].values)),     "NaN dans log_flow !"
assert not np.any(np.isnan(df_volume['log_flow_lag'].values)), "NaN dans log_flow_lag !"

N_v = len(df_volume)
print(f"Étape 2 (Volume)  : {N_v} observations (flow > 0, lag > 0)")
print(f"Taux de zéros (global) : {(df['flow']==0).mean():.1%}")


# =============================================================================
# 7. ENCODAGE DES DYADES ET CLUSTERS
# =============================================================================

# --- Dyades Hurdle ---
dyades_h    = sorted(df_hurdle['dyad'].unique())
dyad_to_h   = {d: i+1 for i, d in enumerate(dyades_h)}
id_to_dyad_h = {v: k for k, v in dyad_to_h.items()}
df_hurdle['dyad_id_h'] = df_hurdle['dyad'].map(dyad_to_h)
D_h = len(dyades_h)

# Cluster continent pour chaque dyade hurdle (1..K_clusters)
# continent_orig est constant pour une dyade donnée → on prend le premier
cluster_h = (
    df_hurdle.groupby('dyad')['continent_orig']
    .first()
    .reindex([id_to_dyad_h[i+1] for i in range(D_h)])
    .values.astype(int)
)

# --- Dyades Volume ---
dyades_v    = sorted(df_volume['dyad'].unique())
dyad_to_v   = {d: i+1 for i, d in enumerate(dyades_v)}
id_to_dyad_v = {v: k for k, v in dyad_to_v.items()}
df_volume['dyad_id_v'] = df_volume['dyad'].map(dyad_to_v)
D_v = len(dyades_v)

cluster_v = (
    df_volume.groupby('dyad')['continent_orig']
    .first()
    .reindex([id_to_dyad_v[i+1] for i in range(D_v)])
    .values.astype(int)
)

print(f"\nDyades Hurdle (D_h) : {D_h}")
print(f"Dyades Volume (D_v) : {D_v}")
print(f"Clusters continent  : {K_clusters}")
print(f"Distribution clusters Volume : {pd.Series(cluster_v).value_counts().sort_index().to_dict()}")


# =============================================================================
# 8. STANDARDISATION DES MATRICES X (Importantes pour les priors)
#    Stan est beaucoup plus efficace quand les covariables sont centrées/réduites.
#    On sauvegarde les stats pour la dé-standardisation si besoin.
# =============================================================================

# Standardisation sur le dataset VOLUME (référence)
X_vol_raw = df_volume[X_VOL_COLS].values.astype(float)
X_h_raw   = df_hurdle[HURDLE_VARS].values.astype(float)

# On standardise les colonnes continues seulement (pas les binaires 0/1)
BINARY_COLS_VOL  = ['LL_i', 'LL_j', 'LB_ij', 'OL_ij', 'COL_ij', 'is_rich_o']
BINARY_COLS_HUR  = ['LB_ij']

def standardize_matrix(X, col_names, binary_cols, fit_stats=None):
    """Standardise les colonnes continues, laisse les binaires intactes."""
    X_std = X.copy().astype(float)
    stats = {}
    for j, col in enumerate(col_names):
        if col not in binary_cols:
            mu = X[:, j].mean() if fit_stats is None else fit_stats[col]['mean']
            sd = X[:, j].std()  if fit_stats is None else fit_stats[col]['std']
            sd = sd if sd > 1e-8 else 1.0  # Éviter division par 0
            X_std[:, j] = (X[:, j] - mu) / sd
            stats[col] = {'mean': mu, 'std': sd}
        else:
            stats[col] = {'mean': 0.0, 'std': 1.0}
    return X_std, stats

X_vol_std, stats_vol = standardize_matrix(X_vol_raw, X_VOL_COLS, BINARY_COLS_VOL)
X_h_std,   stats_h   = standardize_matrix(X_h_raw,  HURDLE_VARS, BINARY_COLS_HUR)

print(f"\nStandardisation OK.")
print(f"  X_vol_std : min={X_vol_std.min():.2f}, max={X_vol_std.max():.2f}")
print(f"  X_h_std   : min={X_h_std.min():.2f},   max={X_h_std.max():.2f}")


# =============================================================================
# 9. DICTIONNAIRE STAN FINAL
# =============================================================================

stan_data = {
    # ---- Partie 1 : HURDLE ----
    'N_h'         : int(N_h),
    'D_h'         : int(D_h),
    'K_h'         : int(K_h),
    'dyad_id_h'   : df_hurdle['dyad_id_h'].astype(int).tolist(),
    'is_mig'      : df_hurdle['is_migration'].astype(int).tolist(),
    'is_mig_lag'  : df_hurdle['is_mig_lag'].astype(float).tolist(),
    'X_h'         : X_h_std.tolist(),
    'cluster_h'   : cluster_h.tolist(),

    # ---- Partie 2 : VOLUME ----
    'N_v'          : int(N_v),
    'D_v'          : int(D_v),
    'K_v'          : int(K_grav),
    'dyad_id_v'    : df_volume['dyad_id_v'].astype(int).tolist(),
    'log_flow'     : df_volume['log_flow'].astype(float).tolist(),
    'log_flow_lag' : df_volume['log_flow_lag'].astype(float).tolist(),
    'X_v'          : X_vol_std.tolist(),
    'cluster_v'    : cluster_v.tolist(),

    # ---- Clusters ----
    'K_clusters'   : int(K_clusters),
}

# Assertions finales de sanité
for key, val in stan_data.items():
    if isinstance(val, list):
        arr = np.array(val, dtype=float)
        if arr.ndim > 0:
            assert not np.any(np.isnan(arr)), f"NaN dans stan_data['{key}'] !"
            assert not np.any(np.isinf(arr)), f"Inf dans stan_data['{key}'] !"

print(f"\n✓ Dictionnaire Stan validé (aucun NaN/Inf).")
print(f"  N_h={N_h}, D_h={D_h}, K_h={K_h}")
print(f"  N_v={N_v}, D_v={D_v}, K_v={K_grav}")
print(f"  K_clusters={K_clusters}")


# =============================================================================
# 10. COMPILATION ET SAMPLING
# =============================================================================

STAN_FILE = "..STAN/M.stan"

print(f"\nCompilation : {STAN_FILE}")
model = CmdStanModel(stan_file=STAN_FILE)
print("Compilation OK.\n")

fit = model.sample(
    data         = stan_data,
    chains       = 4,
    parallel_chains = 4,
    iter_warmup  = 1500,    # Plus long : modèle plus complexe
    iter_sampling = 2000,
    seed         = 42,
    adapt_delta  = 0.95,
    max_treedepth = 12,
    show_progress = True,
)


# =============================================================================
# 11. DIAGNOSTICS
# =============================================================================

print("\n" + "="*65)
print("  DIAGNOSTICS DE CONVERGENCE — ARX Hurdle Géo")
print("="*65)

resume = fit.summary()

# Paramètres globaux à surveiller en priorité
PARAMS_WATCH = [
    # Hurdle
    'alpha_global', 'tau_alpha',
    'beta_lag_global',
    # Gravité
    'mu_intercept', 'tau_mu',
    'phi_global', 'tau_phi',
    # Clusters σ
    'sigma_cluster[1]', 'sigma_cluster[2]', 'sigma_cluster[3]',
    'sigma_cluster[4]', 'sigma_cluster[5]',
]

print(f"\n{'Paramètre':<28} {'Moyenne':>10} {'StdDev':>10} {'R_hat':>8} {'ESS_bulk':>10}")
print("-" * 68)
for p in PARAMS_WATCH:
    if p in resume.index:
        row = resume.loc[p]
        flag = " ✓" if row['R_hat'] < 1.01 else " ⚠️"
        print(f"{p:<28} {row['Mean']:>10.4f} {row['StdDev']:>10.4f} "
              f"{row['R_hat']:>8.4f}{flag} {row['ESS_bulk']:>10.0f}")

rhat_max = resume['R_hat'].max()
ess_min  = resume['ESS_bulk'].min()
print(f"\n  R_hat max    : {rhat_max:.4f}  {'✓ (<1.01)' if rhat_max < 1.01 else '⚠️  (>1.01)'}")
print(f"  ESS_bulk min : {ess_min:.0f}   {'✓ (>400)' if ess_min > 400 else '⚠️  (<400)'}")
print(f"\n{fit.diagnose()}")


# =============================================================================
# 12. VISUALISATIONS
# =============================================================================

idata = az.from_cmdstanpy(
    posterior              = fit,
    log_likelihood         = {'hurdle': 'log_lik_h', 'volume': 'log_lik_v'},
    posterior_predictive   = {'is_mig_hat': 'is_mig_hat', 'log_flow_hat': 'log_flow_hat'},
)

CONTINENT_NAMES = {1:'Europe', 2:'Am.Nord', 3:'Afrique', 4:'Am.Sud', 5:'Asie', 6:'Océanie'}


# --- Figure 1 : Traceplots des hyperparamètres globaux ---
params_trace = ['alpha_global', 'tau_alpha', 'beta_lag_global',
                'mu_intercept', 'phi_global', 'sigma_global']
params_trace = [p for p in params_trace if p in idata.posterior]

fig, axes = plt.subplots(len(params_trace), 2, figsize=(14, 3*len(params_trace)))
fig.suptitle("Traceplots — Hyperparamètres Globaux (ARX Hurdle Géo)",
             fontsize=13, fontweight='bold', y=1.01)
colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']

for i, param in enumerate(params_trace):
    chains_data = idata.posterior[param].values
    ax_t, ax_h = axes[i, 0], axes[i, 1]
    for c in range(chains_data.shape[0]):
        ax_t.plot(chains_data[c], alpha=0.7, lw=0.4, color=colors[c])
    ax_t.set_title(f'Trace : {param}', fontsize=9)
    all_d = chains_data.flatten()
    ax_h.hist(all_d, bins=60, color='#1565C0', alpha=0.8, density=True, edgecolor='none')
    ax_h.axvline(np.mean(all_d), color='red', lw=1.5, linestyle='--',
                 label=f'μ={np.mean(all_d):.3f}')
    ax_h.set_title(f'Posterior : {param}', fontsize=9)
    ax_h.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/traceplots_arx.png', dpi=150, bbox_inches='tight')
plt.close()


# --- Figure 2 : Variance par cluster (Composant C) ---
fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Hétéroscédasticité Géographique — σ par Continent d'Origine",
             fontsize=12, fontweight='bold')

cluster_colors = ['#3F51B5','#F44336','#4CAF50','#FF9800','#9C27B0','#00BCD4']
for k in range(1, K_clusters+1):
    key = f'sigma_cluster[{k}]'
    if key in idata.posterior:
        draws = idata.posterior[key].values.flatten()
        ax.violinplot(draws, positions=[k], widths=0.6, showmedians=True)

ax.set_xticks(range(1, K_clusters+1))
ax.set_xticklabels([CONTINENT_NAMES.get(k, f'C{k}') for k in range(1, K_clusters+1)])
ax.set_ylabel("σ_cluster (volatilité résiduelle)")
ax.set_title("Un σ plus élevé = le modèle est plus incertain sur ce continent")
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/sigma_clusters.png', dpi=150, bbox_inches='tight')
plt.close()


# --- Figure 3 : Coefficients β_grav (forêt de coefficients) ---
if 'beta_grav' in idata.posterior:
    beta_draws = idata.posterior['beta_grav'].values  # (chains, draws, K_v)
    beta_flat  = beta_draws.reshape(-1, K_grav)
    beta_means = beta_flat.mean(axis=0)
    beta_q05   = np.percentile(beta_flat, 5,  axis=0)
    beta_q95   = np.percentile(beta_flat, 95, axis=0)

    order = np.argsort(beta_means)
    fig, ax = plt.subplots(figsize=(10, max(6, K_grav * 0.4)))
    fig.suptitle("Coefficients β de Gravité — IC 90% Posterior",
                 fontsize=12, fontweight='bold')

    colors_coef = ['#F44336' if beta_q05[i] > 0 or beta_q95[i] < 0
                   else '#90A4AE' for i in order]
    ax.barh(range(K_grav), beta_means[order],
            xerr=[beta_means[order]-beta_q05[order], beta_q95[order]-beta_means[order]],
            color=colors_coef, alpha=0.8, ecolor='gray', capsize=3)
    ax.set_yticks(range(K_grav))
    ax.set_yticklabels([X_VOL_COLS[i] for i in order], fontsize=9)
    ax.axvline(0, color='black', lw=1, linestyle='--')
    ax.set_xlabel("Coefficient β (standardisé)")
    ax.set_title("Rouge = IC 90% ne contient pas 0 (effet significatif)", fontsize=10)
    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/beta_grav_forest.png', dpi=150, bbox_inches='tight')
    plt.close()


# --- Figure 4 : PPC Volume ---
if 'log_flow_hat' in idata.posterior_predictive:
    log_flow_hat = idata.posterior_predictive['log_flow_hat'].values.reshape(-1, N_v)
    log_flow_obs = np.array(stan_data['log_flow'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("PPC — Volume (log-flow | flow > 0)", fontsize=12, fontweight='bold')

    axes[0].hist(log_flow_obs, bins=40, color='black', alpha=0.6,
                 density=True, label='Observé', zorder=3)
    for i in range(min(150, log_flow_hat.shape[0])):
        axes[0].hist(log_flow_hat[i], bins=40, alpha=0.02, density=True, color='#2196F3')
    axes[0].set_xlabel("log(flow)")
    axes[0].set_title("Distribution prédite vs observée")
    axes[0].legend()

    y_pred_med = np.median(log_flow_hat, axis=0)
    resid = log_flow_obs - y_pred_med
    mae_v = np.mean(np.abs(resid))
    r2_v  = 1 - np.sum(resid**2) / np.sum((log_flow_obs - log_flow_obs.mean())**2)

    axes[1].scatter(log_flow_obs, y_pred_med, alpha=0.4, s=15, color='#1565C0')
    lim2 = [log_flow_obs.min(), log_flow_obs.max()]
    axes[1].plot(lim2, lim2, 'r--', lw=1.5)
    axes[1].text(0.05, 0.95, f"MAE(log) = {mae_v:.3f}\nR²       = {r2_v:.3f}",
                 transform=axes[1].transAxes, fontsize=9, va='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    axes[1].set_xlabel("log(flow) observé")
    axes[1].set_ylabel("log(flow) prédit (médiane)")
    axes[1].set_title("Observé vs Prédit — ARX Hurdle Géo")
    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/ppc_arx_volume.png', dpi=150, bbox_inches='tight')
    plt.close()

print("\n✓ Toutes les figures sauvegardées.")
print("\n" + "="*65)
print("  RÉSUMÉ FINAL")
print("="*65)
print(f"""
  Composant A (Hurdle) : β_D, β_LB, β_int + α_d hiérarchique
  Composant B (ARX)    : {K_grav} covariables gravité + seuil PIB (β_rich)
  Composant C (Géo)    : σ_cluster[{K_clusters} continents]

  Prochaine étape : extension au dataset complet (190 pays, serveur Onyxia)
  → Remplacer dropna par imputation (KNN ou régression) sur les NaN restants
  → Ajouter variables politiques (Freedom House Index, dummy dictature)
""")